# 16. Quickstart Walkthrough

**What this notebook does:** unpacks `portiere quickstart` into its individual pipeline stages so you can poke at intermediate state.

**What `portiere quickstart` does in one line:** runs the full 5-stage OMOP pipeline against ~20 bundled synthetic patients, fully offline, in ~10 seconds, emitting a reproducibility manifest covering every stage.

This notebook recreates that flow piece by piece. Use it when you want to:
- See what each pipeline stage produces
- Inspect schema / concept mapping outputs interactively
- Read the manifest by hand to understand its structure
- Adapt the same pattern to your own data

## Prerequisites

```bash
pip install portiere-health[polars,quality]
```

No SapBERT download, no Athena needed — everything used here ships in the wheel.

## 1. Locate the bundled demo data

The `portiere._demo_data` module exposes the bundled synthetic CSVs and the Athena-format vocabulary subset. Everything is committed to the wheel.

In [ ]:
from portiere._demo_data import demo_data_dir, vocabulary_dir, synthetic_source_files

print(f"Demo data:  {demo_data_dir()}")
print(f"Vocabulary: {vocabulary_dir()}")
print()
for name, path in synthetic_source_files().items():
    size_kb = path.stat().st_size / 1024
    print(f"  {name:14s}  {path.name:32s}  {size_kb:.1f} KB")

In [ ]:
# Peek at one of the source CSVs — note the deliberately messy column names
import pandas as pd

conditions = pd.read_csv(synthetic_source_files()["conditions"])
print("Columns:", list(conditions.columns))
conditions.head()

Notice the column-name mix:

- `patient_id` — clean (matches the OMOP `person.person_id` source-pattern alias)
- `dx_code` — rephrased (would normally be `diagnosis_code`/`condition_source_value`)
- `onset_date` — abbreviated (alias for `condition_start_date`)

This is the realistic mix Portiere is designed to handle.

## 2. Build the knowledge layer

BM25s lexical index over the bundled ICD-10-CM / LOINC / RxNorm subset. No embeddings needed (offline).

In [ ]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

out = Path.home() / ".cache" / "portiere" / "quickstart_walkthrough"
out.mkdir(parents=True, exist_ok=True)

knowledge_paths = build_knowledge_layer(
    athena_path=str(vocabulary_dir()),
    output_path=str(out / "knowledge_index"),
    backend="bm25s",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
)
knowledge_paths

## 3. Initialize the project

Use the new `with project:` form so the manifest auto-finalizes on exit.

In [ ]:
import portiere
from portiere.config import EmbeddingConfig, KnowledgeLayerConfig, PortiereConfig

config = PortiereConfig(
    local_project_dir=out,
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s", **knowledge_paths),
    # provider="none" lets us run fully offline without sentence-transformers
    embedding=EmbeddingConfig(provider="none"),
)

project = portiere.init(
    name="quickstart-walkthrough",
    target_model="omop_cdm_v5.4",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
    config=config,
)
project

## 4. Run the 5 stages

In [ ]:
with project:
    # Stage 1: ingest — registers the source, captures schema metadata
    source = project.add_source(str(demo_data_dir() / "synthetic_conditions.csv"))
    print(f"Stage 1 ingest: {len(source['columns'])} columns, {source['row_count']} rows")

    # Stage 2: schema mapping — source columns -> OMOP entity.field
    schema_map = project.map_schema(source)
    print(f"Stage 2 schema: {len(schema_map.items)} mappings")

    # Stage 3: concept mapping — source codes -> standard concept_ids
    concept_map = project.map_concepts(source=source)
    print(f"Stage 3 concept: {len(concept_map.items)} concepts mapped")

    # Stage 4: ETL — generate + run the transform
    etl_out = out / "etl_output"
    etl_result = project.run_etl(source, output_dir=str(etl_out))
    print(f"Stage 4 ETL:    output -> {etl_out}")

    # Stage 5: validate — Kahn-style completeness/conformance/plausibility
    val_report = project.validate(output_path=str(etl_out))
    print(f"Stage 5 validate: passed={val_report['all_passed']}, tables={val_report['total_tables']}")

# Manifest finalized on `with` exit

## 5. Inspect outputs

In [ ]:
# Schema mapping — one row per source column
for item in schema_map.items:
    print(f"  {item.source_column:20s} -> {item.target_table}.{item.target_column:30s} "
          f"conf={item.confidence:.2f} status={item.status}")

In [ ]:
# Concept mapping — one row per source code
for item in concept_map.items[:10]:
    print(f"  {item.source_code:10s} -> concept_id={item.target_concept_id} "
          f"({item.target_concept_name[:50]}...) conf={item.confidence:.2f}")

## 6. Read the manifest

Every successful pipeline stage emits a stage entry into `manifest.lock.json`. The manifest is the audit trail and the input to `portiere replay`.

In [ ]:
import json

manifest_path = sorted((out / "quickstart-walkthrough" / "runs").glob("*/manifest.lock.json"))[-1]
manifest = json.loads(manifest_path.read_text())

print(f"Manifest: {manifest_path}")
print(f"  manifest_version: {manifest['manifest_version']}")
print(f"  run_id:           {manifest['run']['run_id']}")
print(f"  duration_s:       {manifest['run']['duration_seconds']}")
print(f"  target_model:     {manifest['target_model']}")
print(f"  embedding:        {manifest['embedding']['name']}")
print(f"  source sha256:    {manifest['source_data']['sha256'][:16]}...")
print(f"  stages: {[s['stage'] for s in manifest['stages']]}")

## 7. Replay

```bash
portiere replay <manifest-path>
```

verifies referenced artifacts and reconstructs the project. See `17_reproducibility_manifest.ipynb` for a deeper walkthrough.

## Next steps

- **Use your own data:** replace the bundled CSV path with your own — the same flow works.
- **Add SNOMED:** download Athena (free with registration) and pass the path to `build_knowledge_layer`. See `docs/documentations/15-vocabulary-setup.md`.
- **Use real embeddings:** drop `embedding=EmbeddingConfig(provider="none")` and the default SapBERT will be used (downloads on first run).
- **Run the benchmark:** `portiere benchmark athena-icd-snomed --athena-dir <your-athena>` — see `docs/benchmarks/athena-icd-snomed.md`.